# Flow Matching on CIFAR-10 — main training notebook

Reproduction réduite de **Lipman et al., *Flow Matching for Generative Modeling*** (ICLR 2023). Ce notebook entraîne **un** modèle. Pour reproduire la Table 1 du papier, l'exécuter trois fois en changeant `PATH_TYPE` dans la cellule de config :

| `PATH_TYPE` | Modèle | Loss | Cible de régression |
|---|---|---|---|
| `"ot"`   | FM avec chemin Optimal Transport | CFM | $u_t(x \mid x_1) = x_1 - (1-\sigma_{\min})x_0$ |
| `"vp"`   | FM avec chemin VP-Diffusion       | CFM | $u_t(x \mid x_1)$ analytique (Th. 3) |
| `"ddpm"` | DDPM (Ho 2020) baseline           | $\epsilon$-matching | $x_0$ (le bruit) |

**Hardware** : conçu pour Colab A100/H100. ~30h par run à 391k steps. Reprise automatique depuis le dernier checkpoint sur Google Drive.

## 1. Configuration

**Change uniquement `PATH_TYPE` entre les 3 runs.** Le reste suit la lignée ADM/Dhariwal-Nichol (le papier sous-spécifie CIFAR-10). Le batch effectif reste 256 via accumulation de gradient.

In [ ]:
PATH_TYPE = "ot"        # "ot" | "vp" | "ddpm"
RUN_NAME = f"fm-{PATH_TYPE}-cifar10"
MAX_STEPS = 391_000     # optimiser steps
BATCH_SIZE = 64         # physical micro-batch (A100 40GB; raise if the smoke test shows headroom)
ACCUM_ITER = 4          # grad-accumulation → effective batch = BATCH_SIZE * ACCUM_ITER = 256
USE_WANDB = True        # set to False if no wandb account
DRIVE_ROOT = "/content/drive/MyDrive/flow-matching-b3"  # Colab Drive checkpoints root
REPO_URL = "https://github.com/Alexis5131/Conditional-Flow-Matching.git"

## 2. Colab setup (skipped on local)

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
print("Colab:", IN_COLAB)

if IN_COLAB:
    from google.colab import drive  # noqa: F401
    drive.mount("/content/drive")
    if not os.path.exists("/content/flow-matching-b3"):
        subprocess.run(["git", "clone", REPO_URL, "/content/flow-matching-b3"], check=True)
    subprocess.run(["pip", "install", "-q", "-e", "/content/flow-matching-b3"], check=True)
    sys.path.insert(0, "/content/flow-matching-b3/src")
    RUN_DIR = f"{DRIVE_ROOT}/{RUN_NAME}"
else:
    sys.path.insert(0, os.path.abspath("../src"))
    RUN_DIR = f"./runs/{RUN_NAME}"

os.makedirs(RUN_DIR, exist_ok=True)
print("RUN_DIR =", RUN_DIR)

In [ ]:
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("vram :", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

## 3. Eval helper — FID (`legacy_tensorflow`, protocole papier)

Le mid-training utilise 10k échantillons (biaisé ↑ vs les 50k de la table finale), tous les `eval_every` steps.

In [ ]:
from pathlib import Path
import torch
from flow_matching_b3.paths import get_path
from flow_matching_b3.sampling import sample
from flow_matching_b3.fid import compute_fid_cifar10
from flow_matching_b3.train import denorm

def make_fid_fn(path, n_samples=10_000, batch=128, solver="dopri5"):
    @torch.no_grad()
    def _fid(model):
        def gen():
            remaining = n_samples
            while remaining > 0:
                b = min(batch, remaining)
                imgs, _ = sample(model, path, shape=(b, 3, 32, 32), solver=solver, device=next(model.parameters()).device)
                yield denorm(imgs)
                remaining -= b
        return compute_fid_cifar10(gen(), n_samples=n_samples)
    return _fid

## 4. Training

In [ ]:
from flow_matching_b3.train import TrainConfig, train

cfg = TrainConfig(
    path_type=PATH_TYPE,
    run_dir=Path(RUN_DIR),
    data_root=Path("/content/data") if IN_COLAB else Path("./data"),
    max_steps=MAX_STEPS,
    batch_size=BATCH_SIZE,
    accum_iter=ACCUM_ITER,
    lr_peak=1e-4,          # ADM default
    warmup_steps=45_000,
    poly_decay=True,
    ema_decay=0.9999,
    ckpt_every=25_000,
    log_every=200,
    eval_every=50_000,
    eval_n_samples=10_000,
    device="cuda" if torch.cuda.is_available() else "cpu",
    dtype="float32",
    use_wandb=USE_WANDB,
    wandb_run_name=RUN_NAME,
)

# Build path object used for in-loop FID eval
if cfg.path_type == "ot":
    eval_path = get_path("ot", sigma_min=cfg.sigma_min)
elif cfg.path_type == "vp":
    eval_path = get_path("vp", beta_min=cfg.vp_beta_min, beta_max=cfg.vp_beta_max)
else:
    eval_path = get_path("ddpm", beta_min=cfg.vp_beta_min, beta_max=cfg.vp_beta_max)

fid_fn = make_fid_fn(eval_path, n_samples=cfg.eval_n_samples)

if USE_WANDB:
    import wandb
    wandb.login()

In [ ]:
# Smoke test mémoire/débit AVANT le run long : régler BATCH_SIZE/ACCUM_ITER pour tenir < 40 GB.
# build() applique déjà TF32 + channels_last + torch.compile → les 1ers pas COMPILENT (lents), on warmup.
import time as _time
from flow_matching_b3.train import build, make_cifar10_loader, _infinite, enable_cuda_perf
from flow_matching_b3.losses import get_loss_fn

if torch.cuda.is_available():
    enable_cuda_perf(tf32=cfg.tf32)
    _m, _opt, _ema, _path = build(cfg)
    _loss_fn = get_loss_fn(_path)
    _it = _infinite(make_cifar10_loader(cfg))
    _mem_fmt = torch.channels_last if cfg.channels_last else torch.contiguous_format
    _m.train()

    def _micro():
        _x, _ = next(_it)
        _x = _x.to(cfg.device, non_blocking=True, memory_format=_mem_fmt)
        _l, _ = _loss_fn(_m, _x, _path, eps=cfg.time_eps)
        (_l / cfg.accum_iter).backward()

    for _ in range(3):  # warmup: déclenche la compilation (non chronométré)
        _micro(); _opt.step(); _opt.zero_grad(set_to_none=True)
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    _N = 12; _t0 = _time.time()
    for _ in range(_N):
        _micro(); _opt.step(); _opt.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    _peak = torch.cuda.max_memory_allocated() / 1e9
    _ips = _N / (_time.time() - _t0)        # micro-batch / s
    _spo = cfg.accum_iter / _ips            # s / pas d'optimiseur
    print(f"physical={cfg.batch_size}  accum={cfg.accum_iter}  -> effective={cfg.batch_size*cfg.accum_iter}")
    print(f"peak VRAM ~ {_peak:.1f} GB / 40 GB")
    print(f"{_ips:.1f} micro-batch/s  ->  {_spo:.3f} s/step  ->  ~{cfg.max_steps*_spo/3600:.0f} h pour {cfg.max_steps} steps")
    print("Astuce: si la VRAM le permet, monte BATCH_SIZE (128) et baisse ACCUM_ITER (2).")
    del _m, _opt, _ema, _it; torch.cuda.empty_cache()
else:
    print("CPU - smoke test sauté")

In [ ]:
final_ckpt = train(cfg, fid_fn=fid_fn)
print("final checkpoint:", final_ckpt)

## 5. Final FID on 50k samples + a sample grid

In [ ]:
from flow_matching_b3.unet import adm_unet_cifar10
from flow_matching_b3.ema import EMA
from flow_matching_b3.train import find_latest_ckpt, make_grid_png
import matplotlib.pyplot as plt

model = adm_unet_cifar10().to(cfg.device)
ema = EMA(model, decay=cfg.ema_decay)
state = torch.load(find_latest_ckpt(Path(RUN_DIR)), map_location=cfg.device)
model.load_state_dict(state["model"])
ema.load_state_dict(state["ema"])

with ema.swap_into(model):
    model.eval()
    final_fid = make_fid_fn(eval_path, n_samples=50_000)(model)
    print(f"final FID(EMA, 50k) for {PATH_TYPE}: {final_fid:.3f}")
    grid_samples, _ = sample(model, eval_path, shape=(64, 3, 32, 32), solver="dopri5", device=cfg.device, seed=0)

grid = make_grid_png(grid_samples, n_row=8).permute(1, 2, 0).cpu()
plt.figure(figsize=(8, 8))
plt.imshow(grid)
plt.axis("off")
plt.title(f"{RUN_NAME} — FID(50k) = {final_fid:.2f}")
plt.savefig(f"{RUN_DIR}/final_samples.png", bbox_inches="tight", dpi=120)
plt.show()